# 🌦️ Plots Climáticos — Juazeiro, BA
**Variáveis:** Pressão · Precipitação · Humidade · Água em Nuvens · Temperatura  
**Período:** 1 dia (24 h) ou 1 semana (7 dias)  
**Fonte:** GFS 0.25° / NOAA · NOMADS  
**Publicação:** @reinanbr

In [1]:
# ─── Dependências ────────────────────────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.lines
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import LineCollection
import matplotlib.ticker as mticker
from datetime import datetime
from IPython.display import display, Image, clear_output


In [2]:
# ─── Conversão de timestamp numpy ────────────────────────────────────────────
def to_dt(v):
    ts = (v - np.datetime64('1970-01-01T00:00')) / np.timedelta64(1, 's')
    return datetime.utcfromtimestamp(float(ts))

VARS_NEEDED = ['prmsl', 'prate', 'r2', 'cwat', 't2m']

# ─── Carrega dados reais ou sintéticos ───────────────────────────────────────
try:
    from noawclg.main import get_noaa_data
    print('⏳ Baixando dados GFS para Juazeiro-BA…')
    _raw_full = {}
    _time_full = _lat = _lon = None
    noaa = get_noaa_data(keys=VARS_NEEDED, date='20260405')
    view = noaa.get_data_from_place('Juazeiro, Bahia, Brazil')
    ds   = view.dataset
    for key in VARS_NEEDED:
        if _time_full is None:
            _time_full = ds['time'].values
            _lat = float(ds['latitude'].values)
            _lon = float(ds['longitude'].values)
        _raw_full[key] = ds[key].values.astype(float)
    # Kelvin → °C
    if _raw_full['t2m'].mean() > 100:
        _raw_full['t2m'] -= 273.15
    DATA_SOURCE = 'GFS 0.25° / NOAA · NOMADS'
    print('✅ Dados GFS carregados!')

except Exception as e:
    print(f'⚠️  Usando dados sintéticos ({e})')
    # Gera série longa (7 dias, step 1h) e fatiaremos conforme período
    _N = 24 * 7 + 1
    _time_full = np.array([
        np.datetime64('2026-04-04T00:00') + np.timedelta64(i, 'h')
        for i in range(_N)
    ])
    np.random.seed(7)
    _h = np.arange(_N)
    _raw_full = {
        'prmsl': 1010 + 3*np.sin(_h/36*np.pi) + np.random.normal(0, 0.4, _N),
        'prate': np.clip(
            np.where(_h % 48 < 6, np.random.exponential(1e-4, _N), 0)
            + np.random.exponential(5e-5, _N)*0.2, 0, None),
        'r2': np.clip(
            55 + 25*np.sin((_h % 24 - 4)*np.pi/12) + np.random.normal(0, 3, _N),
            15, 98),
        'cwat': np.clip(
            0.15 + 0.25*np.abs(np.sin(_h/48*np.pi)) + np.random.exponential(0.04, _N),
            0, None),
        't2m': np.clip(
            30 + 8*np.sin((_h % 24 - 6)*np.pi/12) + np.random.normal(0, 0.8, _N),
            18, 42),
    }
    _lat, _lon = -9.4168, -40.5019
    DATA_SOURCE = 'Dados sintéticos (exemplo)'

print('Dados prontos ✔')

⏳ Baixando dados GFS para Juazeiro-BA…
✅ Dados GFS carregados!
Dados prontos ✔


In [3]:
# ─── Tema ────────────────────────────────────────────────────────────────────
BG       = '#0D0D0D'
CARD     = '#111111'
GRID_C   = '#1A1A1A'
TEXT_PRI = '#F5F0E8'
TEXT_SEC = '#888880'
TEXT_DIM = '#3E3E3A'
HANDLE   = '#FF6B2B'

CMAPS = {
    'prmsl': LinearSegmentedColormap.from_list('pres',  ['#334FBF','#7EB3FF','#E8F4FD'], N=256),
    'prate': LinearSegmentedColormap.from_list('rain',  ['#0D2B45','#1A85C2','#53D8FB','#AAFFF0'], N=256),
    'r2':    LinearSegmentedColormap.from_list('hum',   ['#B45309','#F59E0B','#86EFAC','#22D3EE'], N=256),
    'cwat':  LinearSegmentedColormap.from_list('cloud', ['#1C2333','#475569','#94A3B8','#E2E8F0'], N=256),
    't2m':   LinearSegmentedColormap.from_list('temp',  ['#1E3A5F','#2196F3','#FF9800','#FF3D00','#B71C1C'], N=256),
}
ACCENT = {
    'prmsl':'#7EB3FF', 'prate':'#53D8FB',
    'r2':'#86EFAC',    'cwat':'#94A3B8', 't2m':'#FF9800',
}
ACCENT_HI = {
    'prmsl':'#E8F4FD', 'prate':'#AAFFF0',
    'r2':'#22D3EE',    'cwat':'#E2E8F0', 't2m':'#FF3D00',
}

# configuração por variável (sem 'values' — injetado depois)
VAR_META = {
    'prmsl': dict(label='PRESSÃO ATMOSFÉRICA', title='Pressão Atm.',
                  unit='hPa',    ylabel='Pressão  (hPa)',         fmt='%.0f',  refband=None),
    'prate': dict(label='PRECIPITAÇÃO',        title='Precipitação',
                  unit='mm/h',   ylabel='Taxa  (mm h⁻¹)',         fmt='%.3f',  refband=None, bar_plot=True),
    'r2':    dict(label='HUMIDADE RELATIVA',   title='Humidade Relativa',
                  unit='%',      ylabel='Humidade  (%)',           fmt='%.0f',  refband=(60,80)),
    'cwat':  dict(label='ÁGUA EM NUVENS',      title='Água em Nuvens',
                  unit='kg m⁻²', ylabel='Água em nuvens  (kg m⁻²)', fmt='%.3f', refband=None),
    't2m':   dict(label='TEMPERATURA DO AR',   title='Temperatura 2 m',
                  unit='°C',     ylabel='Temperatura  (°C)',       fmt='%.1f',  refband=(20,30)),
}
print('Tema configurado ✔')

Tema configurado ✔


In [4]:
# ─── Funções auxiliares de plot ───────────────────────────────────────────────
def _tick_dates(dts, t_days, periodo):
    if periodo == '1dia':
        idxs = [i for i,d in enumerate(dts) if d.hour % 3 == 0]
        return [t_days[i] for i in idxs], [dts[i].strftime('%Hh') for i in idxs]
    unique = sorted(set(d.date() for d in dts))
    tdts   = [dts[next(i for i,dd in enumerate(dts) if dd.date()==ud)] for ud in unique[::2]]
    return [(d-dts[0]).total_seconds()/86400 for d in tdts], [d.strftime('%d/%m') for d in tdts]

def _header(fig, label, title, unit, lat, lon, dts, accent, run_str, src, periodo):
    ps = '24 HORAS' if periodo == '1dia' else '7 DIAS'
    fig.text(0.08,0.950, f'{label}  ·  GFS 0.25°  ·  {ps}',
             color=TEXT_SEC, fontsize=8.5, fontfamily='monospace', fontweight='bold', va='bottom')
    fig.text(0.08,0.918, f'Juazeiro — BA  ·  {title}',
             color=TEXT_PRI, fontsize=28, fontweight='black', va='top',
             path_effects=[pe.withStroke(linewidth=5, foreground=BG)])
    fig.text(0.08,0.865,
             f'lat {lat:.2f}°  ·  lon {lon:.2f}°  ·  '
             f'{dts[0].strftime("%d %b %Y")} → {dts[-1].strftime("%d %b %Y")}',
             color=TEXT_SEC, fontsize=8, fontfamily='monospace', va='top')
    fig.add_artist(matplotlib.lines.Line2D([0.08,0.65],[0.858,0.858],
        transform=fig.transFigure, color=accent, linewidth=2.2))

def _footer(fig, src, run_str):
    fig.add_artist(matplotlib.lines.Line2D([0.08,0.96],[0.078,0.078],
        transform=fig.transFigure, color=GRID_C, linewidth=1))
    fig.text(0.08,0.063, f'Fonte: {src}   ·   Gerado: {run_str}',
             color=TEXT_DIM, fontsize=7.5, fontfamily='monospace', va='top')
    fig.text(0.96,0.063, '@reinanbr', color=HANDLE, fontsize=13,
             fontweight='black', fontfamily='monospace', va='top', ha='right',
             path_effects=[pe.withStroke(linewidth=4, foreground=BG)])

def _stats_cards(fig, vals, unit, accent, accent_hi):
    stats = [
        ('MÁXIMA', f'{np.max(vals):.2g} {unit}',  accent_hi),
        ('MÍNIMA', f'{np.min(vals):.2g} {unit}',  accent),
        ('MÉDIA',  f'{np.mean(vals):.2g} {unit}', TEXT_PRI),
        ('DESVIO', f'{np.std(vals):.2g} {unit}',  TEXT_SEC),
    ]
    fig.add_artist(matplotlib.lines.Line2D([0.08,0.96],[0.270,0.270],
        transform=fig.transFigure, color=GRID_C, linewidth=1))
    for (lbl,val,col),x in zip(stats,[0.08,0.30,0.54,0.76]):
        fig.text(x,0.252, lbl, color=TEXT_SEC, fontsize=7.5, fontfamily='monospace',
                 fontweight='bold', va='top')
        fig.text(x,0.224, val, color=col, fontsize=18, fontweight='black', va='top')
        fig.add_artist(matplotlib.lines.Line2D([x,x+0.17],[0.170,0.170],
            transform=fig.transFigure, color=col, linewidth=2.0))

def _grid_lines(ax, vals):
    lo,hi = float(np.min(vals)), float(np.max(vals))
    span  = hi - lo
    step  = max(span/6, 1e-9)
    mag   = 10**np.floor(np.log10(step))
    step  = round(step/mag)*mag
    for y in np.arange(np.floor(lo/step)*step, np.ceil(hi/step)*step+step, step):
        ax.axhline(y, color=GRID_C, linewidth=0.6, zorder=0)

def _colored_line(ax, t_days, vals, cmap):
    lo,hi = float(np.min(vals)), float(np.max(vals))
    pts   = np.array([t_days, vals]).T.reshape(-1,1,2)
    segs  = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc    = LineCollection(segs, cmap=cmap, norm=plt.Normalize(lo,hi),
                           linewidth=2.5, zorder=5, capstyle='round')
    lc.set_array((vals[:-1]+vals[1:])/2)
    ax.add_collection(lc)
    t_i = np.linspace(t_days[0], t_days[-1], 600)
    c_i = np.interp(t_i, t_days, vals)
    nv  = (c_i-lo)/max(hi-lo, 1e-9)
    for k in range(len(t_i)-1):
        ax.fill_between(t_i[k:k+2], lo-(hi-lo)*0.05, c_i[k:k+2],
                        color=cmap(nv[k]), alpha=0.12, linewidth=0, zorder=2)

def _mark_extremes(ax, t_days, vals, accent, accent_hi):
    lo,hi  = float(np.min(vals)), float(np.max(vals))
    span   = hi - lo
    i_hi   = int(np.argmax(vals))
    i_lo   = int(np.argmin(vals))
    def _s(idx): return 1.0 if t_days[idx] < t_days[-1]*0.75 else -3.5
    ax.scatter(t_days[i_hi], vals[i_hi], s=90, color=accent_hi,
               zorder=10, edgecolors=TEXT_PRI, linewidths=1.0)
    ax.annotate(f' MÁX  {vals[i_hi]:.3g}', xy=(t_days[i_hi],vals[i_hi]),
                xytext=(t_days[i_hi]+_s(i_hi), vals[i_hi]+span*0.08),
                color=accent_hi, fontsize=8, fontweight='bold', fontfamily='monospace',
                zorder=11, arrowprops=dict(arrowstyle='-', color=accent_hi, lw=0.7))
    ax.scatter(t_days[i_lo], vals[i_lo], s=90, color=accent,
               zorder=10, edgecolors=TEXT_PRI, linewidths=1.0)
    ax.annotate(f' MÍN  {vals[i_lo]:.3g}', xy=(t_days[i_lo],vals[i_lo]),
                xytext=(t_days[i_lo]+_s(i_lo), vals[i_lo]-span*0.12),
                color=accent, fontsize=8, fontweight='bold', fontfamily='monospace',
                zorder=11, arrowprops=dict(arrowstyle='-', color=accent, lw=0.7))

print('Funções carregadas ✔')

Funções carregadas ✔


In [5]:
# ─── Função principal de geração ──────────────────────────────────────────────
def gerar_plots(periodo, vars_sel, salvar):
    """Gera e exibe os plots no notebook.
    periodo  : '1dia' | '1semana'
    vars_sel : lista de chaves ex ['prmsl','t2m']
    salvar   : bool — salva PNGs no disco
    """
    N_HORAS = 24 if periodo == '1dia' else 24*7
    STEP_H  = 1  if periodo == '1dia' else 3

    # fatia da série temporal
    step_td = np.timedelta64(STEP_H, 'h')
    idxs    = np.arange(0, N_HORAS+1, STEP_H)
    # garante não ultrapassar o tamanho real
    idxs    = idxs[idxs < len(_time_full)]

    time_vals = _time_full[idxs]
    raw = {k: _raw_full[k][idxs] for k in VARS_NEEDED}
    raw['prate_mmh'] = raw['prate'] * 3600.0

    dts    = [to_dt(v) for v in time_vals]
    t_days = np.array([(d-dts[0]).total_seconds()/86400 for d in dts])

    run_str      = datetime.utcnow().strftime('%Y-%m-%dT%H:%MZ')
    tick_x, tick_lbl = _tick_dates(dts, t_days, periodo)

    VALUES_MAP = {
        'prmsl': raw['prmsl'],
        'prate': raw['prate_mmh'],
        'r2':    raw['r2'],
        'cwat':  raw['cwat'],
        't2m':   raw['t2m'],
    }

    suffix  = '1d' if periodo == '1dia' else '7d'
    outputs = []

    for key in vars_sel:
        cfg      = VAR_META[key]
        vals     = VALUES_MAP[key]
        cmap     = CMAPS[key]
        accent   = ACCENT[key]
        acc_hi   = ACCENT_HI[key]
        bar_plot = cfg.get('bar_plot', False)

        fig = plt.figure(figsize=(13, 16), dpi=110)
        fig.patch.set_facecolor(BG)

        ax = fig.add_axes([0.08, 0.29, 0.88, 0.50])
        ax.set_facecolor(CARD)

        _header(fig, cfg['label'], cfg['title'], cfg['unit'],
                _lat, _lon, dts, accent, run_str, DATA_SOURCE, periodo)

        lo = float(np.min(vals))
        hi = float(np.max(vals))
        ax.set_ylim(lo-(hi-lo)*0.12, hi+(hi-lo)*0.18)
        _grid_lines(ax, vals)

        if cfg.get('refband'):
            ax.axhspan(*cfg['refband'], color=accent, alpha=0.07, zorder=1)

        if bar_plot:
            norm_v = (vals-lo)/max(hi-lo, 1e-9)
            colors = [cmap(v) for v in norm_v]
            bar_w  = (t_days[1]-t_days[0])*0.85
            ax.bar(t_days, vals, width=bar_w, color=colors, alpha=0.85, zorder=4, linewidth=0)
            mean_v = float(np.mean(vals))
            ax.axhline(mean_v, color=accent, linewidth=1.2, linestyle='--',
                       alpha=0.6, zorder=6, label=f'Média  {mean_v:.3g} mm/h')
            ax.legend(loc='upper right', framealpha=0, labelcolor=TEXT_SEC, fontsize=8)
        else:
            _colored_line(ax, t_days, vals, cmap)
            _mark_extremes(ax, t_days, vals, accent, acc_hi)

        # estilo dos eixos
        ax.set_xlim(t_days[0], t_days[-1])
        ax.set_xticks(tick_x)
        ax.set_xticklabels(tick_lbl, color=TEXT_SEC, fontsize=8, fontfamily='monospace')
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(cfg['fmt']))
        ax.tick_params(axis='y', colors=TEXT_SEC, labelsize=8.5)
        ax.tick_params(axis='x', colors=TEXT_DIM, length=3)
        for sp in ax.spines.values(): sp.set_edgecolor(GRID_C)
        ax.set_ylabel(cfg['ylabel'], color=TEXT_SEC, fontsize=8.5,
                      fontfamily='monospace', labelpad=10)

        _stats_cards(fig, vals, cfg['unit'], accent, acc_hi)
        _footer(fig, DATA_SOURCE, run_str)

        # renderiza para bytes e exibe inline
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=110, bbox_inches='tight',
                    facecolor=BG, edgecolor='none')
        buf.seek(0)
        display(Image(data=buf.read()))
        buf.seek(0)

        if salvar:
            fname = f'juazeiro_ba_{key}_{suffix}.png'
            fig.savefig(fname, dpi=130, bbox_inches='tight',
                        facecolor=BG, edgecolor='none')
            outputs.append(fname)
            print(f'💾  Salvo: {fname}')

        plt.close(fig)

    if salvar and outputs:
        print(f'\n✅  {len(outputs)} arquivo(s) salvo(s):', outputs)

print('Função gerar_plots() pronta ✔')

Função gerar_plots() pronta ✔


In [6]:
# ─── Interface interativa ─────────────────────────────────────────────────────

# --- Widgets ---
w_periodo = widgets.ToggleButtons(
    options=[('📅  1 Dia', '1dia'), ('📆  1 Semana', '1semana')],
    value='1semana',
    description='Período:',
    style={'button_width': '140px', 'description_width': '70px'},
    layout=widgets.Layout(margin='8px 0'),
)

VAR_LABELS = [
    ('🌡️  Temperatura (t2m)',      't2m'),
    ('🌧️  Precipitação (prate)',    'prate'),
    ('💧  Humidade Relativa (r2)',  'r2'),
    ('🌬️  Pressão Atm. (prmsl)',   'prmsl'),
    ('☁️  Água em Nuvens (cwat)',  'cwat'),
]

w_vars = widgets.SelectMultiple(
    options=VAR_LABELS,
    value=['t2m', 'prate', 'r2', 'prmsl', 'cwat'],
    description='Variáveis:',
    rows=5,
    layout=widgets.Layout(width='340px'),
    style={'description_width': '80px'},
)

w_salvar = widgets.Checkbox(
    value=False,
    description='Salvar PNGs no disco',
    indent=False,
    layout=widgets.Layout(margin='8px 0'),
)

w_btn = widgets.Button(
    description='▶  Gerar Plots',
    button_style='success',
    layout=widgets.Layout(width='160px', height='38px', margin='10px 0'),
)

w_out = widgets.Output()

hint = widgets.HTML(
    '<span style="color:#888;font-size:12px">'  
    'Ctrl+Click (ou Cmd+Click) para selecionar múltiplas variáveis.</span>'
)

def on_click(b):
    vars_sel = list(w_vars.value)
    if not vars_sel:
        with w_out:
            clear_output()
            print('⚠️  Selecione pelo menos uma variável.')
        return
    with w_out:
        clear_output(wait=True)
        print(f'⏳ Gerando {len(vars_sel)} plot(s) — {w_periodo.value}…')
        gerar_plots(w_periodo.value, vars_sel, w_salvar.value)

w_btn.on_click(on_click)

painel = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 12px">🌦️ Plots Climáticos — Juazeiro, BA</h3>'),
    w_periodo,
    widgets.HBox([widgets.VBox([w_vars, hint]), widgets.VBox([w_salvar, w_btn])],
                 layout=widgets.Layout(gap='24px', align_items='flex-start')),
])

display(painel, w_out)

NameError: name 'widgets' is not defined